# Zvec Walkthrough

**Zvec** is a lightweight, high-performance, in-process vector database for developers who need fast and reliable similarity search without operating a separate database service.

In this notebook, you will build a small multi-modal image search demo. You will create a collection, load pre-generated image embeddings, write documents to Zvec, optimize the collection, and query it with both text and image inputs.

This walkthrough is intentionally self-contained: the JSONL files included with the notebook already contain sample images and embeddings, so you can focus on the Zvec workflow first.

Useful references:

- New to vector search? Start with the [Concepts](https://zvec.org/en/docs/db/concepts/) section.
- Need API details or more examples? See the [Zvec documentation](https://zvec.org/en/docs/db/).

## Table of Contents

1. [Installation](#installation)
1. [Global Configuration](#global-configuration)
1. [Creating a Collection](#creating-a-collection)
1. [Writing Data](#writing-data)
1. [Querying with Vector Search](#querying-with-vector-search)
1. [Next Steps](#next-steps)
1. [Optional Cleanup](#optional-cleanup)

## Installation

Zvec is available on [PyPI](https://pypi.org/project/zvec/) as a standard Python package. In a Jupyter notebook, use `%pip` so the package is installed into the same Python environment used by the current kernel.

```python
%pip install zvec
```

In [ ]:
%pip install zvec

In [ ]:
# Verify the installation by importing zvec.
import zvec

print("Zvec version:", zvec.__version__)

## Global Configuration

Before using Zvec, configure logging for the notebook session. Here we send logs to the console and show warnings or more severe messages only, which keeps the output readable while still surfacing important issues.

In [ ]:
zvec.init(log_type=zvec.LogType.CONSOLE, log_level=zvec.LogLevel.WARN)

## Creating a Collection

In Zvec, data is stored in **collections**. A collection is similar to a table in a relational database: it has a schema, stores documents, and owns its indexing configuration.

The next cell creates a small demo collection for image search. Zvec stores the collection on disk, so we first choose a local directory:

- `DEMO_COLLECTION_DIR`: the directory where Zvec stores the collection files for this walkthrough.

The directory is disposable: the cell resets it before creating the collection, which lets you rerun the notebook from the top without hitting a "path already exists" error.

The collection schema mirrors the useful fields in `data.jsonl`:

- a scalar field, `base64_image`, used to display the image;
- an indexed nullable scalar field, `category`, used for category facets and filtered search;
- a vector field, `embedding`, used for similarity search.

For the full collection creation guide, see the [documentation](https://zvec.org/en/docs/db/collections/create/).

In [ ]:
from pathlib import Path
import shutil

# Zvec stores each collection as a local directory.
# This directory is only for the disposable notebook demo.
DEMO_COLLECTION_DIR = Path("./image_search")

# If this cell is rerun in the same kernel, remove the previous demo collection
# through Zvec first.
if "collection" in globals():
    try:
        collection.destroy()
    except Exception:
        pass

# Remove any leftover demo directory before creating a fresh collection.
# Do not delete collection directories like this in production code.
if DEMO_COLLECTION_DIR.exists():
    shutil.rmtree(DEMO_COLLECTION_DIR)

# Define scalar fields using FieldSchema.
image_encoding = zvec.FieldSchema(
    name="base64_image",
    data_type=zvec.DataType.STRING,
)

image_category = zvec.FieldSchema(
    name="category",
    data_type=zvec.DataType.STRING,
    nullable=True,
    index_param=zvec.InvertIndexParam(enable_range_optimization=False),
)

# Define the vector field using VectorSchema.
image_embedding = zvec.VectorSchema(
    name="embedding",
    data_type=zvec.DataType.VECTOR_FP32,
    dimension=1024,
    index_param=zvec.HnswIndexParam(metric_type=zvec.MetricType.COSINE),
)

# Define the collection schema.
collection_schema = zvec.CollectionSchema(
    name="image_search",
    fields=[image_encoding, image_category],
    vectors=[image_embedding],
)

# Create and open the collection in the demo directory.
collection = zvec.create_and_open(
    path=str(DEMO_COLLECTION_DIR),
    schema=collection_schema,
)

print("Created a fresh Zvec collection directory for this walkthrough.")
print(f"- storage directory: {DEMO_COLLECTION_DIR}")
print("- scalar fields: base64_image, category")
print("- indexed field for filtering: category")
print("- vector field: embedding")

You should now see a new directory named **image_search** in your working directory. That directory contains the collection data and index files for this demo.

> **Important**
>
> The automatic reset above is only for this tutorial. In a real application, use a stable collection directory and open existing data with `zvec.open()`. See [Open a Collection](https://zvec.org/en/docs/db/collections/open/) for details.

We use an [HNSW index](https://zvec.org/en/docs/db/concepts/vector-index/hnsw-index/) for the `embedding` vector field. HNSW is a strong default for low-latency approximate nearest-neighbor search.

Always align the vector index settings with the embedding model. In this example, the embeddings are 1024-dimensional and are intended for cosine similarity, so the vector schema uses `dimension=1024` and `MetricType.COSINE`.

Inspect the schema to confirm the collection was created with the fields and vector settings we expect:

In [ ]:
print("Collection schema:")
print(collection.schema)

For a detailed explanation of the schema format, see [Inspect a Collection](https://zvec.org/en/docs/db/collections/inspect/#collection-schema).

The collection is empty at this point. Check the stats before writing data:

In [ ]:
print("Collection stats:")
print(collection.stats)
# doc_count should be 0 because the collection is empty.

## Writing Data

A typical vector search workflow starts with raw data, such as images or text. An [embedding model](https://zvec.org/en/docs/db/concepts/vector-embedding/#what-is-an-embedding-model) converts that raw data into a [vector embedding](https://zvec.org/en/docs/db/concepts/vector-embedding/). You can also store scalar fields next to the vectors, such as labels, titles, categories, or timestamps, so queries can combine semantic search with filtering.

To keep this notebook self-contained, the included **JSONL** files contain a small pre-generated dataset. The source images come from [ImageNet-Val5k](https://modelscope.cn/datasets/iic/imagenet-val5k-image), and the embeddings were precomputed with the **Qwen2.5-VL-Embedding** multi-modal model.

Each record in `data.jsonl` includes:

1. `id`: a unique document identifier.
1. `image_encoding`: the original image encoded as a base64 string.
1. `image_embedding`: a precomputed vector embedding for the image.
1. `category`: an optional label such as `animal` or `vehicle`.

Before loading the data, install the small display dependencies used by the helper functions.

In [ ]:
%pip install Pillow matplotlib

### Load and Inspect the Data

Load the sample data and inspect one record before writing anything to Zvec.

In [ ]:
import walkthrough_utils

# Load the pre-generated sample data using the provided utility.
data = walkthrough_utils.load_jsonl("./data.jsonl")

The `data` variable is a `list[dict]`. Each dictionary represents one document with these keys:

1. `id`
1. `image_encoding`
1. `image_embedding`
1. `category` (optional)

Let's inspect one sample.

In [ ]:
# Inspect the structure and size of the dataset.
print("Structure of the first document:", data[0].keys())
print(f"Total number of documents loaded: {len(data)}")

# Preview an example document. Change this index to explore other samples.
example_index = 2
raw_doc = data[example_index]

print(f"\nDisplaying image[{example_index}] from the dataset:")
walkthrough_utils.display_image_from_base64(raw_doc["image_encoding"])

print(f"ID: {raw_doc['id']}")
print(f"Image embedding dimension: {len(raw_doc['image_embedding'])}")
print("Image embedding preview:", raw_doc["image_embedding"][:6], "...")
if "category" in raw_doc:
    print("Category:", raw_doc["category"])

### Upsert Documents into the Collection

Now convert each raw record into a Zvec `Doc` and write it to the collection with `upsert()`.

A `Doc` is the basic unit stored in a collection. It has three main parts:

1. `id`: a unique string identifier.
1. `fields`: scalar values keyed by field name.
1. `vectors`: vector values keyed by vector field name.

Both `fields` and `vectors` must match the schema created earlier. In this demo, each document stores `base64_image`, optionally stores `category`, and always stores the `embedding` vector.

We use `upsert()` instead of `insert()` here. `insert()` rejects duplicate document IDs, so rerunning this cell after the documents have already been written could fail. `upsert()` keeps the cell safe to rerun: if a document ID already exists, it overwrites that document; otherwise, it creates a new one.

For more detail, see [Documents](https://zvec.org/en/docs/db/concepts/data-modeling/#documents).

In [ ]:
docs = []

for raw_doc in data:
    fields = {
        "base64_image": raw_doc["image_encoding"],
    }
    if "category" in raw_doc:
        fields["category"] = raw_doc["category"]

    doc = zvec.Doc(
        id=raw_doc["id"],
        fields=fields,
        vectors={
            "embedding": raw_doc["image_embedding"],
        },
    )
    docs.append(doc)

statuses = collection.upsert(docs)
failed = [status for status in statuses if not status.ok()]

if failed:
    print(f"Upserted {len(docs) - len(failed)} documents; {len(failed)} failed.")
    print(failed[:3])
else:
    print(f"Upserted {len(docs)} documents.")

The cell above uses [`upsert()`](https://zvec.org/en/docs/db/data-operations/upsert/) to keep the notebook idempotent. This is helpful in Jupyter, where rerunning individual cells is common.

In production code, choose the write method that matches your data contract. Use [`insert()`](https://zvec.org/en/docs/db/data-operations/insert/) when duplicate document IDs should be treated as an error. Use `upsert()` when reruns, refreshes, or replacement writes are expected.

### Optimize the Collection

Newly written vectors are first staged in a lightweight flat buffer for fast ingestion. Calling [`optimize()`](https://zvec.org/en/docs/db/collections/optimize/) asks Zvec to build the configured vector index from those staged vectors.

For this tiny demo, brute-force search would already be fast. Still, calling `optimize()` here shows the normal workflow you would use after larger batch writes.

In [ ]:
collection.optimize()
print(collection.stats)

## Querying with Vector Search

Now that the documents are written to the collection, we can retrieve semantically similar images with vector search.

The image embeddings in this demo were generated with **Qwen2.5-VL-Embedding**. Because it is a multi-modal model, images and text live in a shared semantic space: you can query image embeddings with either an image or a text description.

To keep the notebook self-contained, the query embeddings are also precomputed and stored in JSONL files.

### Text Query

For a text query such as "cute dog", a production application would usually:

1. encode the text with the same embedding model;
1. pass the resulting vector to Zvec;
1. retrieve the nearest image vectors.

Here, we skip the encoding step and load the precomputed query vector. Each text query record contains:

- `text`: the original query string;
- `embedding`: the corresponding vector representation.

In [ ]:
# Load the pre-generated text queries using the provided utility.
text_queries = walkthrough_utils.load_jsonl("./text_queries.jsonl")

print(f"Total number of text queries loaded: {len(text_queries)}")
print("Structure of the first text query:", text_queries[0].keys())

query_index = 0
query = text_queries[query_index]

print(f"\nText query at index {query_index}:")
print("Text:", query["text"])
print("Embedding dimension:", len(query["embedding"]))
print("Embedding preview:", query["embedding"][:6], "...")

The first text query is **"Cute dog"**. Use its precomputed embedding to search the collection:

In [ ]:
print(f"Text query: {text_queries[0]['text']}")

result = collection.query(
    zvec.Query(
        field_name="embedding",
        vector=text_queries[0]["embedding"],
    ),
    topk=3,
    include_vector=False,
)

print("\nQuery result:\n")
print(result)

The query returns the top 3 matching documents. Each result is a `Doc` containing:

- `id`: the document ID written earlier;
- `fields`: scalar fields, including `base64_image` and sometimes `category`;
- `vectors`: vector values, which are empty here because `include_vector=False`.

Next, decode each returned `base64_image` field and display the actual images.

In [ ]:
for idx, doc in enumerate(result, start=1):
    category = doc.fields.get("category", "uncategorized")
    print(f"\nDisplaying image #{idx} with ID: {doc.id}")
    print(f"Category: {category}")
    walkthrough_utils.display_image_from_base64(doc.fields["base64_image"])

The top results should look visually related to the query. Now define a helper function so you can try other text queries by changing `query_index`.

In [ ]:
def display_similar_images_by_text_query(query_index: int):
    text_query = text_queries[query_index]
    print(f"Text query[{query_index}]: {text_query['text']}")

    result = collection.query(
        zvec.Query(
            field_name="embedding",
            vector=text_query["embedding"],
        ),
        topk=3,
        include_vector=False,
    )

    for idx, doc in enumerate(result, start=1):
        category = doc.fields.get("category", "uncategorized")
        print(f"\nDisplaying image #{idx} with ID: {doc.id}")
        print(f"Category: {category}")
        walkthrough_utils.display_image_from_base64(doc.fields["base64_image"])

Now try a more specific text query: **"Find images that have both a dog and a car."**

In [ ]:
# Change query_index to explore other text queries.
display_similar_images_by_text_query(query_index=1)

> **Interpreting the results**
>
> With this small sample dataset, the second and third results may not perfectly match the text query. That is expected: there is only one image in the demo data that clearly contains both a dog and a car. In production-scale collections with many more candidates, vector search usually has more relevant neighbors to choose from.

### Filtered Vector Search with a Category Facet

In a real image search application, a user often combines two inputs:

1. a broad natural-language search, such as `cute`;
1. a structured UI facet, such as the `animal` category.

The vector query handles the fuzzy visual intent. The scalar filter enforces the category selected by the user.

In [ ]:
from collections import Counter

category_counts = Counter(doc.get("category", "uncategorized") for doc in data)

print("Available category facets:")
for category, count in category_counts.most_common():
    print(f"- {category}: {count}")

Now run the same `cute` search twice: first without a filter, then with `category = 'animal'`.

The filtered query is the product behavior you would expect from a faceted search UI: keep the semantic ranking, but only consider documents from the selected category.

In [ ]:
cute_query = text_queries[16]

unfiltered_results = collection.query(
    zvec.Query(
        field_name="embedding",
        vector=cute_query["embedding"],
    ),
    topk=3,
    include_vector=False,
)

animal_results = collection.query(
    zvec.Query(
        field_name="embedding",
        vector=cute_query["embedding"],
    ),
    filter="category = 'animal'",
    topk=3,
    include_vector=False,
)

print(f"Search intent: {cute_query['text']}")

print("\nUnfiltered results:")
for idx, doc in enumerate(unfiltered_results, start=1):
    category = doc.fields.get("category", "uncategorized")
    print(f"\nResult #{idx}: {doc.id}")
    print(f"Category: {category}")
    walkthrough_utils.display_image_from_base64(doc.fields["base64_image"])

print("\nFiltered results where category = 'animal':")
for idx, doc in enumerate(animal_results, start=1):
    category = doc.fields.get("category", "uncategorized")
    print(f"\nResult #{idx}: {doc.id}")
    print(f"Category: {category}")
    walkthrough_utils.display_image_from_base64(doc.fields["base64_image"])

This is a **filtered vector search**: Zvec searches by vector similarity, but only among documents that satisfy the scalar filter.

For larger collections, indexing commonly filtered fields such as `category` helps Zvec evaluate those constraints efficiently. For more details, see [Hybrid search](https://zvec.org/en/docs/db/data-operations/query/hybrid/).

### Image Query

Text is not the only possible query input. Because the embedding model is multi-modal, we can also search with an image.

The included `image_queries.jsonl` file contains several precomputed image queries. Each record includes:

- `encoding`: the query image encoded as a base64 string;
- `embedding`: the corresponding image embedding.

Load the image queries and run the same vector search workflow.

In [ ]:
# Load the pre-generated image queries using the provided utility.
image_queries = walkthrough_utils.load_jsonl("./image_queries.jsonl")

print(f"Loaded {len(image_queries)} image queries.")


def display_similar_images_by_image_query(query_index: int):
    image_query = image_queries[query_index]

    print(f"Image query[{query_index}]:")
    walkthrough_utils.display_image_from_base64(image_query["encoding"])

    result = collection.query(
        zvec.Query(
            field_name="embedding",
            vector=image_query["embedding"],
        ),
        topk=3,
        include_vector=False,
    )

    print(f"\nRetrieved {len(result)} similar images:")
    for idx, doc in enumerate(result, start=1):
        category = doc.fields.get("category", "uncategorized")
        print(f"\nDisplaying image #{idx} with ID: {doc.id}")
        print(f"Category: {category}")
        walkthrough_utils.display_image_from_base64(doc.fields["base64_image"])

> **Try another image query**
>
> Change `query_index` in the next cell to explore different query images. The top result should usually be closest to the query image, while later results may be looser matches because the demo dataset is small.

In [ ]:
# Change query_index to explore other image queries.
display_similar_images_by_image_query(query_index=0)

## Next Steps

This notebook covered the core Zvec workflow: create a collection, write documents, optimize the index, and query by vector similarity.

Good next reads:

- [Quickstart](https://zvec.org/en/docs/db/quickstart/) for a compact end-to-end database workflow.
- [AI-friendly docs](https://zvec.org/en/docs/db/ai-friendly/) for copyable context you can give to coding assistants.
- [Schema evolution](https://zvec.org/en/docs/db/collections/schema-evolution/) for changing fields and indexes after a collection already exists.
- [Multi-vector search](https://zvec.org/en/docs/db/data-operations/query/multi-vector/) for querying multiple embedding spaces and fusing the results.

## Optional Cleanup

When you are done experimenting, you can delete the demo collection to free local storage.

The operation below is **permanent**: it deletes the collection directory and all data associated with it. Run it only when you no longer need this demo collection.

In [ ]:
# Optional cleanup: permanently delete the demo collection.
collection.destroy()

print("Collection deleted successfully.")